This is a markdown cell. It describes the purpose of this notebook and its code.

This notebook will transform a PDF scan of the glossary into an Excel glossary for further use in the translation of manorial records from Latin to English. 

Last updated by Kuba Kowalski on 30/11/2025.

In [5]:
# Install for convenience
!pip install ocrmypdf pdfplumber python-docx pymupdf requests PyPDF2

!sudo apt-get install -y tesseract-ocr tesseract-ocr-lat


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app


In [6]:
from pathlib import Path
import subprocess
from PyPDF2 import PdfReader  # pip install PyPDF2
import csv

## OCR - Free

In [7]:
# Paths

glossary_pdf = Path(
    r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\Glossary_Cuxham.pdf"
)

output_dir = Path(
    r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\glossary\Cuxham"
)
output_dir.mkdir(parents=True, exist_ok=True)

ocr_pdf = output_dir / f"{glossary_pdf.stem}_ocr.pdf"
raw_txt_path = output_dir / "Glossary_Cuxham_ocr.txt"
csv_path = output_dir / "glossary_cuxham_raw.csv"

In [ ]:
# OCR (no output)

print(f"OCR processing input: {glossary_pdf} -> {ocr_pdf}")

cmd = [
    "ocrmypdf",
    "--force-ocr",
    "-l", "lat+eng",
    "--output-type", "pdf",   # normal PDF, no strict PDF/A validation -> If validation set to Y, then fatal error due to page 27. 
    "--optimize", "0",        # skip image optimization -> If set to Y, then fatal error due to page 27. 
    str(glossary_pdf),
    str(ocr_pdf),
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("Return code:", result.returncode)
if result.stdout:
    print("STDOUT:\n", result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)

if result.returncode not in (0, 4):
    raise RuntimeError(
        f"ocrmypdf failed with code {result.returncode}. "
        f"Fix this before continuing."
    )

print(f"OCR finished; proceeding with {ocr_pdf}")


OCR processing input: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\Glossary_Cuxham.pdf -> C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\glossary\Cuxham\Glossary_Cuxham_ocr.pdf
Return code: 0
STDERR:
 Start processing 27 pages concurrently
    3 [tesseract] lots of diacritics - possibly poor OCR
   21 [tesseract] lots of diacritics - possibly poor OCR
    7 [tesseract] lots of diacritics - possibly poor OCR
    5 [tesseract] lots of diacritics - possibly poor OCR
Postprocessing...
Image optimization ratio: 1.00 savings: 0.0%
Total file size ratio: 0.94 savings: -6.0%

OCR finished; proceeding with C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\glossary\Cuxham\Glossary_Cuxham_ocr.pdf


In [9]:
# OCR with output txt

reader = PdfReader(str(ocr_pdf))
all_text = []
for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    all_text.append(page_text)

full_text = "\n".join(all_text)
raw_txt_path.write_text(full_text, encoding="utf-8")
print(f"Saved raw OCR text to: {raw_txt_path}")


Saved raw OCR text to: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\glossary\Cuxham\Glossary_Cuxham_ocr.txt


### Evaluation of OCR results

Missing terms due to inferior scan quality and subsequent errors in the OCR. Example: Page 2/776 is missing top-most paragraph for "agnus" meaning lamb. Must be added manually. 

Solution: Abandon OCRmyPDF and Tesseract. Use OpenAI API instead due to superior results. For example, ChatGPT 4o correctly transcribes the entire paragraph on page 2/776 for agnus/lamb. 

## OCR with OpenAI API (ChatGPT 4o)

In [10]:
# Install for convenience
!pip install openai

!pip install tiktoken

!pip install pdf2image


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# Packages
from pathlib import Path
from openai import OpenAI
import tiktoken
from pdf2image import convert_from_path
import subprocess
import re
from PIL import Image
import base64
import pandas as pd

In [12]:
# Path to your key file
key_path = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\open-ai-key.txt")

# Read key from file
with open(key_path, "r", encoding="utf-8") as f:
    api_key = f.read().strip()

# Create OpenAI client using this key
client = OpenAI(api_key=api_key)

print("Loaded API key from file ✅")

try:
    client.models.list()
    print("VALID API KEY")
except Exception as e:
    print("INVALID API KEY")
    print(e)

Loaded API key from file ✅
VALID API KEY


### Conversion of PDF to PNG

In [14]:
# Conversion of PDF glossary to images (required for effective OCR by OpenAI)

# Input PDF
glossary_pdf = Path(
    r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\Glossary_Cuxham.pdf"
)

# Output folders
image_dir = glossary_pdf.parent / "images_extracted"
image_dir.mkdir(exist_ok=True)

print("Extracting pages as images...")

# Convert each page to PNG at 400 DPI (higher = better OCR)
pages = convert_from_path(str(glossary_pdf), dpi=400)

image_paths = []

for i, page in enumerate(pages, start=1):
    img_path = image_dir / f"page_{i:03d}.png"
    page.save(img_path, "PNG")
    image_paths.append(img_path)

print(f"Extracted {len(image_paths)} pages to {image_dir}")


Extracting pages as images...
Extracted 27 pages to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\images_extracted


### OpenAI OCR cost estimation

In [15]:
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_text_tokens(text: str) -> int:
    return len(enc.encode(text))


def estimate_image_tokens(image_path: Path):
    """
    Estimate GPT-4o-mini image tokens BEFORE uploading.

    Notes:
    - GPT-4o typically downsamples images to ~512–768 px long side.
    - Tokens correlate with resized pixel count / ~230.
    - Estimate error ~10–20% but good enough for budgeting.
    """

    img = Image.open(image_path)
    w, h = img.size

    # OpenAI-style downsampling (approx. rules)
    longest_side_target = 768   # typical GPT-4o internal size
    scale = longest_side_target / max(w, h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    # Pixel count after downsampling
    pixels = new_w * new_h

    # Empirical: ~1 token per 230–260 pixels for GPT-4o-mini
    est_tokens = int(pixels / 240)

    return {
        "orig_size": (w, h),
        "scaled_size": (new_w, new_h),
        "approx_image_tokens": est_tokens
    }


# FULL PRE-CALL TOKEN ESTIMATE FOR A PAGE
def estimate_total_tokens_before_call(image_path: Path):
    instruction = "Transcribe ALL text in this image exactly. Do NOT translate. Preserve line breaks."
    text_tokens = count_text_tokens(instruction)

    img_info = estimate_image_tokens(image_path)

    total_estimate = text_tokens + img_info["approx_image_tokens"]

    return {
        "instruction_text": instruction,
        "text_tokens": text_tokens,
        "image_tokens_est": img_info["approx_image_tokens"],
        "total_est_tokens": total_estimate,
        "orig_image_size": img_info["orig_size"],
        "scaled_image_size": img_info["scaled_size"]
    }

In [ ]:
# Estimation of token costs per page, multiply by 27 for full costs

image_path = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\images_extracted\page_002.png")

est = estimate_total_tokens_before_call(image_path)
est

{'instruction_text': 'Transcribe ALL text in this image exactly. Do NOT translate. Preserve line breaks.',
 'text_tokens': 17,
 'image_tokens_est': 1833,
 'total_est_tokens': 1850,
 'orig_image_size': (3306, 4431),
 'scaled_image_size': (573, 768)}

In [33]:
openai_output_dir = glossary_pdf.parent / "openai_ocr_txt"
openai_output_dir.mkdir(exist_ok=True)

def ocr_with_openai(image_path: Path) -> str:
    # Load and encode image
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    b64 = base64.b64encode(img_bytes).decode("utf-8")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a high-accuracy OCR engine. Transcribe all visible text exactly. This is permitted OCR."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Extract all text exactly and preserve line breaks."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{b64}"
                        }
                    }
                ]
            }
        ],
        max_tokens=4096
    )

    return response.choices[0].message.content

In [ ]:
# Check to see if OCR works to avoid wasting tokens
print(ocr_with_openai(image_paths[2])[:800])

```
WORD LIST WITH GLOSSARY

68 (wethers), 70 (harvest exp.), 78 (geese), 80 (malt), etc.). Hand xxx once writes 'Autumn
pni' 85 (harvest exp.).
autumpnum: harvest. Hand xvi writes
atumpno abl. s. 90, but 'autumpno' abl. s. 91. See
also autumpn-.
auxilium: help. ausilio abl. s. 57 (petty exp.).
avena: oats. For hands ix, xiii, xxx, extended to pl. (MSS.: 'auraenum' gen. pl. 29, 32, 38;
'Auene' nom. pl. 74, 81, 85 (oats in each); 'omnes Auen' nom. pl. 81 (oats)). For other
hands extended to sing. Hand xii writes 'auenis' abl. pl. 62 (barley), but 'auen' gen. s. 63
(sale of dredge).
averar' trans. verb; infin.: to cart (as a customary service), 86. 
axo 1st conj. to fit with axes. and, gen. masc. or neut. abl. s. 30 (costs of carts), per abl. pl.
57 (costs of carts). For hand xxiii extended 


#### IMPORTANT: Cell below costs money

Don't re-run it willy-nilly.

In [35]:
# OCR (runtime is 20 min for 27 pages, costs 0.30 EUR)
openai_text_files = []

for img_path in image_paths:
    text = ocr_with_openai(img_path)
    out_path = openai_output_dir / (img_path.stem + ".txt")
    out_path.write_text(text, encoding="utf-8")
    openai_text_files.append(out_path)
    print(f"OCR'd with OpenAI: {img_path.name} → {out_path.name}")

print("OpenAI OCR complete.")

OCR'd with OpenAI: page_001.png → page_001.txt
OCR'd with OpenAI: page_002.png → page_002.txt
OCR'd with OpenAI: page_003.png → page_003.txt
OCR'd with OpenAI: page_004.png → page_004.txt
OCR'd with OpenAI: page_005.png → page_005.txt
OCR'd with OpenAI: page_006.png → page_006.txt
OCR'd with OpenAI: page_007.png → page_007.txt
OCR'd with OpenAI: page_008.png → page_008.txt
OCR'd with OpenAI: page_009.png → page_009.txt
OCR'd with OpenAI: page_010.png → page_010.txt
OCR'd with OpenAI: page_011.png → page_011.txt
OCR'd with OpenAI: page_012.png → page_012.txt
OCR'd with OpenAI: page_013.png → page_013.txt
OCR'd with OpenAI: page_014.png → page_014.txt
OCR'd with OpenAI: page_015.png → page_015.txt
OCR'd with OpenAI: page_016.png → page_016.txt
OCR'd with OpenAI: page_017.png → page_017.txt
OCR'd with OpenAI: page_018.png → page_018.txt
OCR'd with OpenAI: page_019.png → page_019.txt
OCR'd with OpenAI: page_020.png → page_020.txt
OCR'd with OpenAI: page_021.png → page_021.txt
OCR'd with Op

In [36]:
# Merging all TXT files of individual into one. 

# Folder where the OCR files are stored
openai_output_dir = glossary_pdf.parent / "openai_ocr_txt"

# Output file
merged_output = openai_output_dir / "merged_glossary.txt"

# Collect all .txt files
txt_files = sorted(openai_output_dir.glob("page_*.txt"))

print(f"Found {len(txt_files)} OCR text files.")

# Merge
with open(merged_output, "w", encoding="utf-8") as outfile:
    for i, txt_file in enumerate(txt_files, start=1):
        text = txt_file.read_text(encoding="utf-8")

        # Write a clean, consistent page separator
        outfile.write(f"\n\n----- PAGE {i} ({txt_file.name}) -----\n\n")
        outfile.write(text)
        outfile.write("\n")

print(f"MERGED OUTPUT SAVED TO:\n{merged_output}")

Found 27 OCR text files.
MERGED OUTPUT SAVED TO:
C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\openai_ocr_txt\merged_glossary.txt


## Transform to Excel Glossary

In [45]:
# Path to your merged glossary text file
txt_path = Path(
    r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\openai_ocr_txt\merged_glossary_adjusted_for_input.txt"
)

# Output CSV path
csv_path = txt_path.with_name("glossary_cuxham_parsed.csv")

GRAMMAR_TOKENS = {
    "nom.", "acc.", "abl.", "gen.", "dat.", "voc.",
    "pl.", "s.", "sg.", "p.",
    "m.", "f.", "n."
}


def is_page_separator(line: str) -> bool:
    return line.strip().startswith("----- PAGE ")


def is_header_line(line: str) -> bool:
    return line.strip().upper().startswith("WORD LIST WITH GLOSSARY")


def is_head_line(line: str) -> bool:
    """
    Treat a line as a head entry only if:
    - it has a colon
    - the FIRST colon is NOT inside parentheses
    """
    if not line.strip():
        return False
    if is_page_separator(line) or is_header_line(line):
        return False

    stripped = line.lstrip()
    if ":" not in stripped:
        return False

    first_colon = stripped.index(":")
    first_paren = stripped.find("(")

    # If there's a "(" before the first ":", colon is inside (...) → not a head line
    if 0 <= first_paren < first_colon:
        return False

    return True


def split_head_latin_and_grammar(head: str):
    tokens = head.strip().split()
    if not tokens:
        return "", "", ""

    first_gram_idx = None
    for i, tok in enumerate(tokens):
        base = tok.rstrip(",")
        if base in GRAMMAR_TOKENS:
            first_gram_idx = i
            break

    if first_gram_idx is None:
        latin = head.strip()
        return latin, "", ""

    lemma_tokens = tokens[:first_gram_idx]
    tail_tokens = tokens[first_gram_idx:]

    # Is tail pure grammar or a complex bundle?
    tail_simple = True
    for tok in tail_tokens:
        base = tok.rstrip(",")
        if base and base not in GRAMMAR_TOKENS:
            tail_simple = False
            break

    latin = " ".join(lemma_tokens).strip()

    if tail_simple:
        grammar = " ".join(t.rstrip(",") for t in tail_tokens).strip()
        head_extra = ""
    else:
        grammar = ""
        head_extra = " ".join(tail_tokens).strip()

    return latin, grammar, head_extra


def split_head_line(line: str):
    head, rest = line.split(":", 1)
    latin, grammar, head_extra = split_head_latin_and_grammar(head)
    rest = rest.strip()

    english = ""
    notes_from_rest = ""

    if rest:
        m = re.match(r"([^\.]*\.)\s*(.*)", rest, flags=re.DOTALL)
        if m:
            english_sentence = m.group(1).strip()
            english = english_sentence.rstrip(".").strip()
            notes_from_rest = m.group(2).strip()
        else:
            english = rest.strip()

    notes_parts = []
    if head_extra:
        notes_parts.append(head_extra.strip())
    if notes_from_rest:
        notes_parts.append(notes_from_rest.strip())
    notes_head = " ".join(notes_parts).strip()

    return latin, grammar, english, notes_head


entries = []
current = None

with txt_path.open(encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.rstrip("\n")

        if not line.strip():
            continue
        if is_page_separator(line) or is_header_line(line):
            continue

        if is_head_line(line):
            if current:
                entries.append(current)

            latin, grammar, english, notes_head = split_head_line(line)
            current = {
                "latin": latin,
                "grammar": grammar,
                "english": english,
                "notes": notes_head,
            }
        else:
            if current:
                cont = line.strip()
                if current["notes"]:
                    current["notes"] += " " + cont
                else:
                    current["notes"] = cont
            else:
                # Orphan continuation; ignore
                pass

if current:
    entries.append(current)

print(f"Parsed {len(entries)} entries from {txt_path.name}")

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["latin", "grammar", "english", "notes"])
    for e in entries:
        writer.writerow([e["latin"], e["grammar"], e["english"], e["notes"]])

print(f"Glossary CSV written to: {csv_path}")

# Output Excel path
excel_path = csv_path.with_suffix(".xlsx")

# Load CSV
df = pd.read_csv(csv_path)

# Save to Excel
df.to_excel(excel_path, index=False)

print(f"Excel version created:\n{excel_path}")

# Optional: auto-adjust column widths for readability
from openpyxl import load_workbook

wb = load_workbook(excel_path)
ws = wb.active

for column in ws.columns:
    max_length = 0
    column_letter = column[0].column_letter
    for cell in column:
        try:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        except:
            pass
    ws.column_dimensions[column_letter].width = min(max_length + 2, 60)  # cap width for very long notes

wb.save(excel_path)
print("Column widths adjusted for readability.")

Parsed 647 entries from merged_glossary_adjusted_for_input.txt
Glossary CSV written to: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\openai_ocr_txt\glossary_cuxham_parsed.csv
Excel version created:
C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\glossary\openai_ocr_txt\glossary_cuxham_parsed.xlsx
Column widths adjusted for readability.
